# Governance Dashboard — QuickSight Analysis for Drift Monitoring

> 📊 **Note**: this notebook contains interactive output cells (Evidently reports, plotly charts). They render fully in JupyterLab; static viewers like the GitHub/GitLab renderer strip the JavaScript.


This notebook programmatically creates a complete QuickSight dashboard — no manual UI steps.

**Schema awareness:** the feature-drift dataset (Sheet 3) and the inference dataset both expose `monitoring_run_id`, and the datasets join on it rather than on a time window. Lab 5D stamps the run id onto every inference row it scored, so run membership is recorded at scoring time instead of being inferred from timestamps. The join is therefore exact at any cadence: a run whose window opened weeks ago still matches its own rows, and back-to-back runs minutes apart cannot double-count, because each inference row carries exactly one run's id however much the runs' time ranges overlap.

**Visuals — Sheet 1 (Model Drift Trends):**
1. ROC-AUC: Baseline vs Current Over Time
2. Model Performance Metrics Over Time
3. ROC-AUC Trend by Model Version
4. Model-Drift Verdict Rate by Model Package ARN
5. Model-Drift Verdict Rate by Endpoint
6. Performance by Training Snapshot
7. Model Lineage Audit (per monitoring run)
8. Latest Current ROC-AUC (KPI)
9. Confusion Matrix Over Time (TP/FP/TN/FN)
10. ROC-AUC Degradation % Over Time
11. Ground-Truth Coverage Over Time (labels / predictions)

**Visuals — Sheet 2 (Data Drift Trends):**
1. Data Drift Share Over Time
2. Drifted Features Count Over Time
3. Drift Alerts Timeline
4. Data Drift Share by Model Version
5. Data Drift Share by Endpoint
6. Inference Volume vs Drift Share Correlation
7. Latest Data Drift Share (KPI)
8. Source Data — monitoring_responses (table)
9. Prediction Score Distribution Over Time (leading indicator)
10. Drift Verdict Sample Size Per Run

**Visuals — Sheet 3 (Feature Drift Trends):**
1. Feature Drift Score Timeline (per-feature line chart)
2. Top 15 Most-Drifting Features (All Time)
3. Drift Severity Distribution by Feature (Top 15)
4. Feature Drift Heatmap (Features × Time)
5. Feature Drift Details (per run × feature table)
6. Highest Current Drift Score (KPI)
7. Feature Drift Heatmap (Features × Model Version)
8. Max Feature Drift Score Per Run (worst-feature signal)
9. Repeat-Offender Features (# of runs where drift_detected=TRUE)
10. Raw drift_score — p-value tests (KS / Chi-square) — LOWER = more drift
11. Raw drift_score — distance tests (Wasserstein / Jensen-Shannon / PSI) — HIGHER = more drift

## What this notebook does

This notebook builds (or refreshes) the QuickSight **governance dashboard** — the
inference, drift, feature-drift, and prediction-accuracy visuals — on top of the
Athena tables written by the drift-monitoring pipeline.

**All of the logic lives in one place:** `src/governance/create_governance_dashboard.py`.
That module is the single source of truth for every dataset, Athena view, visual, analysis,
and dashboard definition. It is also what `main.py dashboard create` calls. This notebook is
a thin driver over that module — it does **not** redefine any datasets or visuals inline, so
the notebook and the CLI can never drift out of sync.

> Because everything is defined in the module, the feature-drift visuals automatically use the
> test-agnostic **`drift_magnitude`** field (higher = more drift, regardless of which statistical
> test Evidently picked) instead of the ambiguous raw `drift_score`.

## 1. Setup & Configuration

In [ ]:
# Load the shared environment written once by lab0-setup/setup.ipynb, put src/
# on sys.path, and import the governance module. lab0-setup is the single
# source of truth — no per-lab discovery or hardcoding.
import os, sys
from pathlib import Path
from dotenv import load_dotenv

import boto3

# Find the repo-root .env (written by lab0-setup/setup.ipynb) and load it.
project_root = Path.cwd()
_env_dir = next((p for p in [project_root, *project_root.parents]
                 if (p / '.env').exists()), None)
if _env_dir is None:
    raise RuntimeError('No .env found. Run lab0-setup/setup.ipynb first.')
load_dotenv(_env_dir / '.env', override=True)

# Force lab5/ (this dir) onto sys.path so `src/` imports resolve, and drop any
# stale 'src' modules from a previous run.
_lab5_str = str(project_root)
if _lab5_str not in sys.path:
    sys.path.insert(0, _lab5_str)
for _k in [k for k in list(sys.modules.keys()) if k == 'src' or k.startswith('src.')]:
    del sys.modules[_k]

# Config-driven identifiers (single source of truth: src/config/config.py).
from src.config.config import AWS_DEFAULT_REGION, QUICKSIGHT_IDENTITY_REGION
import src.governance.create_governance_dashboard as gov

PROJECT_NAME = os.environ.get('PROJECT_NAME', 'bank-marketing-prediction')
ACCOUNT_ID = os.environ.get('ACCOUNT_ID') or \
    boto3.client('sts', region_name=AWS_DEFAULT_REGION).get_caller_identity()['Account']

print(f'\u2713 Loaded shared environment from {_env_dir / ".env"}')
print(f'  PROJECT_NAME             = {PROJECT_NAME}')
print(f'  ACCOUNT_ID               = {ACCOUNT_ID}')
print(f'  AWS_DEFAULT_REGION       = {AWS_DEFAULT_REGION}')
print(f'  QUICKSIGHT_IDENTITY_REGION = {QUICKSIGHT_IDENTITY_REGION}')


> ⚠️ **QuickSight subscription required (once per AWS account).**
>
> The account must be subscribed to QuickSight **Enterprise** before this notebook can create anything — the Definition API it publishes through is an Enterprise feature. The next cell reports the edition it finds.
>
> Setup has two halves, and they are separate events on purpose:
>
> 1. **subscribe the account**, which is what creates the `aws-quicksight-service-role-v0` service role;
> 2. **grant that role access to the data**, by deploying with `AttachQuickSightServiceRolePolicy=true` (`QuickSightServiceRoleS3Policy` / `QuickSightServiceRoleAthenaPolicy` in `templates/2-iam.yaml`).
>
> The order is forced: an IAM policy resource fails outright if the role it targets does not exist, which is why the parameter defaults to `false`. And skipping the second half is the usual cause of a dashboard that publishes successfully but whose every panel fails to load — QuickSight reads Athena and S3 **as its own service role**, not as this notebook's execution role.
>
> Two commands tell you where you stand:
>
> ```bash
> ACCOUNT_ID="$(aws sts get-caller-identity --query Account --output text)"
>
> aws quicksight describe-account-settings --aws-account-id "${ACCOUNT_ID}" \
>   --region us-east-1 --query 'AccountSettings.Edition' --output text
>
> aws iam list-role-policies --role-name aws-quicksight-service-role-v0 \
>   --query 'PolicyNames' --output text
> ```
>
> You want `ENTERPRISE` from the first and both `QuickSightS3DataLakeAccess` and `QuickSightAthenaAccess` from the second. If you get both, there is nothing to do — run the next cell.
>
> **Half 1 — subscribe, from the console.** Search the console for `quick`: you'll get two entries, **Amazon Quick** (the suite the BI product now lives in) and **QuickSight** (the classic entry) — either works. On the form, set a globally unique account name and a notification email, and leave **Authentication method** on its default (*Password-based or Single-Sign On*, the equivalent of `IAM_AND_QUICKSIGHT`) and **Encryption** on the AWS-managed key. **Default region** there is where QuickSight stores its own data, which is not the same thing as the identity region. The form has **no edition selector**, so re-run the first check afterwards rather than assuming — `STANDARD` has no Definition API and upgrades from *Manage QuickSight → Account settings*.
>
> **Half 2 — grant the service role access.** With the role now in existence, flip the stack parameter. Only the IAM nested stack changes, so this takes minutes rather than a full redeploy:
>
> ```bash
> STACK_NAME="${PROJECT_NAME:-bank-marketing-prediction}-workshop"
>
> OVERRIDES=""
> for k in $(aws cloudformation describe-stacks --stack-name "${STACK_NAME}" \
>              --query 'Stacks[0].Parameters[].ParameterKey' --output text); do
>   if [ "${k}" = "AttachQuickSightServiceRolePolicy" ]; then
>     OVERRIDES="${OVERRIDES} ParameterKey=${k},ParameterValue=true"
>   else
>     OVERRIDES="${OVERRIDES} ParameterKey=${k},UsePreviousValue=true"
>   fi
> done
>
> aws cloudformation update-stack --stack-name "${STACK_NAME}" \
>   --use-previous-template \
>   --capabilities CAPABILITY_NAMED_IAM CAPABILITY_AUTO_EXPAND \
>   --parameters ${OVERRIDES}
> ```
>
> The workshop docs (`model-governance-at-scale-with-sagemaker-ai`, Lab 5E → *Before You Start*) walk both halves with screenshots and are the canonical version.
>
> **Automating both halves instead.** For provisioning accounts in bulk rather than working through the labs, `SUBSCRIBE_QUICKSIGHT=true QUICKSIGHT_NOTIFICATION_EMAIL=you@example.com ./scripts/deploy-workshop.sh` subscribes, waits for provisioning, confirms the service role, and deploys with the parameter set. It defaults to `false` because the subscription is billable and permanent.
>
> Two things this notebook never needs, but you do if you want to open the dashboard in the QuickSight UI yourself: a QuickSight **Author/Admin** user for your identity (*Manage QuickSight → Manage users*), and `QUICKSIGHT_IDENTITY_REGION=<region>` in `.env` if your account's identity region is not `us-east-1`.
>
> Full prerequisites + troubleshooting: see the README's **QuickSight prerequisites (one-time per account)** section.

## 2. Verify QuickSight Subscription

In [ ]:
# Same check gov.create_dashboard() runs internally, surfaced here so the two
# things that can block this notebook are named before anything is created.
# Both are account/IAM setup rather than notebook problems, so this cell
# reports and continues instead of raising.
quicksight_admin = boto3.client('quicksight', region_name=QUICKSIGHT_IDENTITY_REGION)

subscription = gov.check_quicksight_subscription(
    account_id=ACCOUNT_ID, quicksight_admin_client=quicksight_admin,
)
QUICKSIGHT_READY = subscription['subscribed']

if QUICKSIGHT_READY:
    print(f"✓ QuickSight active (Edition: {subscription['edition']})")
    if subscription['edition'] == 'STANDARD':
        print('  ⚠ Definition API requires Enterprise edition — upgrade before Section 3.')
elif subscription['reason'] == 'access-denied':
    print('✗ Cannot check QuickSight: this role lacks quicksight:DescribeAccountSettings.')
    print('  QuickSight may well be subscribed — this is an IAM gap, not a sign-up gap.')
    print('  Fix: run scripts/deploy-workshop.sh to apply the QuickSightGovernanceDashboard')
    print('       statement from templates/2-iam.yaml to this notebook\'s execution role.')
else:
    print('✗ QuickSight not subscribed for this account.')
    print('  Subscribe from the console (once per account, billable): search the AWS')
    print('  console for "quick" and sign up. The form has no edition selector, so')
    print('  re-run this cell afterwards to confirm you landed on ENTERPRISE.')
    print('  Then attach the data-access policies to aws-quicksight-service-role-v0 —')
    print('  see the markdown cell above for both halves.')


## 3. Verify Inference Data in Athena

In [ ]:
# Build the entire dashboard in one shot.
# gov.create_dashboard() is self-contained -- it creates its own boto3 clients,
# resolves the account, checks QuickSight subscription, verifies Athena data,
# creates all datasets/views/analysis/dashboard, and returns a summary dict.
#
# This is the same entry point that `main.py dashboard create` uses.
#
# Nothing below can be fixed from the notebook: QuickSight has to be signed up
# for once per account, and the execution role has to be allowed to call it. So
# a failure is reported and the notebook continues \u2014 the drift data it would
# visualise is already in Athena and MLflow either way (Lab 5D).
from botocore.exceptions import ClientError

result = None
try:
    result = gov.create_dashboard(region=AWS_DEFAULT_REGION)
except gov.QuickSightUnavailable as exc:
    print(f'\u2717 Dashboard not created \u2014 {exc}')
except ClientError as exc:
    if exc.response['Error']['Code'] not in ('AccessDeniedException', 'UnrecognizedClientException'):
        raise
    # Subscription check passed but a later call was denied: the role has some
    # QuickSight permissions but not all of them.
    print(f'\u2717 Dashboard not created \u2014 QuickSight denied a call this notebook needs:')
    print(f'  {exc.operation_name}: {exc.response["Error"]["Message"]}')
    print('  Fix: attach the full quicksight:* action list from templates/2-iam.yaml')
    print('       (QuickSightGovernanceDashboard statement) to this notebook\'s execution role.')

if result:
    print()
    print('=' * 70)
    print('Dashboard build complete')
    print('=' * 70)
    print(f"QuickSight subscribed: {result['quicksight_subscribed']} (edition: {result['quicksight_edition']})")
    print(f"Dashboard URL:  {result['dashboard_url']}")
    print(f"Dashboard ARN:  {result['dashboard_arn']}")
    print(f"Analysis ARN:   {result['analysis_arn']}")
    print()
    print('Datasets:')
    for k in ('inference', 'drift', 'feature_drift', 'feature_level', 'accuracy'):
        arn_key = f'{k}_dataset_arn'
        print(f"  {k:14s} {result.get(arn_key, 'N/A')}")
    if result.get('embed_url'):
        print(f"\nEmbed URL (valid 10h):\n  {result['embed_url']}")

## 9. Publish Dashboard via Definition API

In [ ]:
CONFIRM_DELETE = False  # set True to delete all governance QuickSight resources

if CONFIRM_DELETE:
    outcome = gov.delete_dashboard(region=AWS_DEFAULT_REGION)
    print('Deleted:  ', outcome['deleted'])
    print('Not found:', outcome['not_found'])
    print('Errors:   ', outcome['errors'])
else:
    print('Cleanup skipped — set CONFIRM_DELETE = True to run.')